# 01 - Dataset Download, Subset Sampling & n-FPS Frame Extraction

Builds the evaluation subset for the adaptive-semantic VLM study from two video QA benchmarks:

| Dataset | Hugging Face repo | Size (full) | Format |
|---|---|---|---|
| **MVBench** | [`OpenGVLab/MVBench`](https://huggingface.co/datasets/OpenGVLab/MVBench) | ~17 GB of zips, 20 tasks x 200 QAs | multiple choice, answer given as option text |
| **Video-MME** | [`lmms-eval/Video-MME`](https://huggingface.co/datasets/lmms-eval/Video-MME) | ~100 GB in 20 chunk zips, 900 videos / 2,700 QAs | multiple choice A-D |

### Pipeline

1. **Annotations**: MVBench task JSONs + Video-MME parquet (a few MB).
2. **Zip index**: read each zip's central directory *remotely* (HTTP range requests, nothing large is downloaded).
3. **Subset sampling**: `N_PER_CATEGORY` questions per category per dataset, one question per video.
4. **Subset video download**: pull *only* the sampled videos out of the remote zips (set `DOWNLOAD_FULL = True` to mirror the full datasets instead).
5. **Calibration split**: `CALIB_PER_CATEGORY` extra questions per category on videos *not* in the test subset
   (used only to tune the stopping threshold of the proposed method).
6. **Frame extraction** at a per-dataset `FPS` -> `data/frames/fps_<n>/<dataset>/<frame_key>/`.

### Output layout

```text
data/
  raw/mvbench/json/*.json                    annotations
  raw/videomme/videomme/*.parquet            annotations (+ subtitle.zip)
  subset/manifest.jsonl  (+ .csv)            one row per sampled test question
  subset/manifest_calib.jsonl  (+ .csv)      calibration questions (disjoint videos)
  videos/<dataset>/<zip member path>         subset videos
  frames/fps_5/mvbench/<frame_key>/frame_00000.jpg ... + frames.json
  frames/fps_1/videomme/<frame_key>/...
```

`frames.json` holds the timestamp of every saved frame. Notebook `02` reads it.

In [ ]:
# Run once in the kernel's environment if packages are missing.
# %pip install -U "huggingface_hub>=0.34" pandas pyarrow opencv-python tqdm matplotlib

In [ ]:
import os
import json
import random
import shutil
import zipfile
import threading
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
try:
    from dotenv import load_dotenv
    load_dotenv(Path.cwd() / ".env")          # picks up HF_TOKEN if set (faster Hub downloads)
except ImportError:
    pass

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from huggingface_hub import HfFileSystem, snapshot_download

print("opencv", cv2.__version__, "| pandas", pd.__version__)

## 1. Configuration

In [ ]:
# ---------------------------------------------------------------- paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # run from Github/ or Github/notebooks/
DATA_ROOT = PROJECT_ROOT / "data"
RAW_ROOT = DATA_ROOT / "raw"
SUBSET_DIR = DATA_ROOT / "subset"
VIDEO_ROOT = DATA_ROOT / "videos"
FRAMES_ROOT = DATA_ROOT / "frames"

# ---------------------------------------------------------------- subset
SEED = 42
N_PER_CATEGORY = 35                       # questions (= distinct videos) per category, per dataset
CALIB_PER_CATEGORY = 10                   # calibration questions per category, per dataset (0 = skip)
DATASETS = ["mvbench", "videomme"]
VIDEOMME_DURATIONS = ["short", "medium"]  # short < 2 min, medium 4-15 min, long 30-60 min

# ---------------------------------------------------------------- download
DOWNLOAD_FULL = False                     # True -> mirror every zip first (~17 GB + ~100 GB)
DOWNLOAD_WORKERS = 4

# ---------------------------------------------------------------- frames
# frames per second to keep, per dataset (fractional values work: 0.5 = one frame every 2 s).
# MVBench clips are 5-60 s, so they get a denser pool; Video-MME runs up to ~15 min.
FPS = {"mvbench": 5.0, "videomme": 1.0}
USE_MVBENCH_BOUNDS = True                 # some MVBench tasks only ask about the [start, end] segment of the video
FRAME_MAX_SIDE = None                     # e.g. 720 to downscale saved frames; None keeps native resolution
JPEG_QUALITY = 95
EXTRACT_WORKERS = 4

def fps_for(dataset):
    return FPS[dataset] if isinstance(FPS, dict) else FPS


for d in (RAW_ROOT, SUBSET_DIR, VIDEO_ROOT, FRAMES_ROOT):
    d.mkdir(parents=True, exist_ok=True)

### Category mapping

The five study categories are mapped to the native task labels of each benchmark. Edit the lists to change the mix.
Within a category, samples are drawn round-robin across the listed tasks, so each task is represented about equally.

MVBench tasks **not** used: `fine_grained_pose` (NTU RGB+D videos are not redistributed in the HF repo) and
`episodic_reasoning` (stored as pre-extracted 3 FPS TVQA frames, not videos, and relies on subtitles).

In [ ]:
CATEGORIES = ["temporal_reasoning", "object_recognition", "counting", "spatial_reasoning", "event_understanding"]

MVBENCH_TASKS = {
    "temporal_reasoning":  ["action_sequence", "action_prediction"],
    "object_recognition":  ["object_existence", "object_interaction"],
    "counting":            ["action_count", "moving_count"],
    "spatial_reasoning":   ["moving_direction", "object_shuffle", "egocentric_navigation"],
    "event_understanding": ["unexpected_action", "scene_transition", "state_change", "counterfactual_inference"],
}

VIDEOMME_TASKS = {
    "temporal_reasoning":  ["Temporal Reasoning"],
    "object_recognition":  ["Object Recognition"],
    "counting":            ["Counting Problem"],
    "spatial_reasoning":   ["Spatial Reasoning"],
    "event_understanding": ["Action Reasoning"],
}

# Folder of each MVBench task inside the repo's video zips (same table as the official VideoChat2 evaluation).
# The first path component is the zip name. A few fixed videos live under "data0613/<same prefix>".
MVBENCH_PREFIX = {
    "action_sequence":          "star/Charades_v1_480/",
    "action_prediction":        "star/Charades_v1_480/",
    "action_antonym":           "ssv2_video/",
    "fine_grained_action":      "Moments_in_Time_Raw/videos/",
    "unexpected_action":        "FunQA_test/test/",
    "object_existence":         "clevrer/video_validation/",
    "object_interaction":       "star/Charades_v1_480/",
    "object_shuffle":           "perception/videos/",
    "moving_direction":         "clevrer/video_validation/",
    "action_localization":      "sta/sta_video/",
    "scene_transition":         "scene_qa/video/",
    "action_count":             "perception/videos/",
    "moving_count":             "clevrer/video_validation/",
    "moving_attribute":         "clevrer/video_validation/",
    "state_change":             "perception/videos/",
    "character_order":          "perception/videos/",
    "egocentric_navigation":    "vlnqa/",
    "counterfactual_inference": "clevrer/video_validation/",
}
MVBENCH_BOUNDED = {"action_sequence", "action_prediction", "object_interaction", "action_localization"}

REPOS = {"mvbench": "OpenGVLab/MVBench", "videomme": "lmms-eval/Video-MME"}

unknown = [t for ts in MVBENCH_TASKS.values() for t in ts if t not in MVBENCH_PREFIX]
assert not unknown, f"no zip prefix for MVBench tasks: {unknown}"

## 2. Download annotations (and optionally the full datasets)

In [ ]:
snapshot_download(REPOS["mvbench"], repo_type="dataset", local_dir=RAW_ROOT / "mvbench",
                  allow_patterns=["json/*.json", "README.md"])
snapshot_download(REPOS["videomme"], repo_type="dataset", local_dir=RAW_ROOT / "videomme",
                  allow_patterns=["videomme/*.parquet", "subtitle.zip", "README.md"])
print("annotations ready under", RAW_ROOT)

In [ ]:
# Optional: mirror every video zip. Everything below reads local zips when they exist, otherwise streams the
# needed members straight from the Hub, so this is only needed if you want the complete datasets on disk.
if DOWNLOAD_FULL:
    snapshot_download(REPOS["mvbench"], repo_type="dataset", local_dir=RAW_ROOT / "mvbench",
                      allow_patterns=["video/*.zip", "video/*.txt"], max_workers=DOWNLOAD_WORKERS)
    snapshot_download(REPOS["videomme"], repo_type="dataset", local_dir=RAW_ROOT / "videomme",
                      allow_patterns=["videos_chunked_*.zip"], max_workers=DOWNLOAD_WORKERS)
else:
    print("DOWNLOAD_FULL = False -> only the sampled videos will be downloaded (section 5)")

## 3. Load annotations into one table

Common schema: `qid, dataset, category, task, video, question, options (list), answer (letter), start, end, duration`.

In [ ]:
LETTERS = "ABCDEFGH"


def load_mvbench():
    rows = []
    for category, tasks in MVBENCH_TASKS.items():
        for task in tasks:
            items = json.loads((RAW_ROOT / "mvbench" / "json" / f"{task}.json").read_text(encoding="utf-8"))
            for i, it in enumerate(items):
                bounded = task in MVBENCH_BOUNDED and "start" in it and "end" in it
                rows.append(dict(
                    qid=f"mvb_{task}_{i:03d}", dataset="mvbench", category=category, task=task,
                    video=it["video"], question=it["question"], options=list(it["candidates"]),
                    answer=LETTERS[it["candidates"].index(it["answer"])],
                    start=float(it["start"]) if bounded else None,
                    end=float(it["end"]) if bounded else None,
                    duration=None,
                ))
    return pd.DataFrame(rows)


def load_videomme():
    df = pd.read_parquet(next((RAW_ROOT / "videomme" / "videomme").glob("*.parquet")))
    task_to_cat = {t: c for c, ts in VIDEOMME_TASKS.items() for t in ts}
    df = df[df.task_type.isin(task_to_cat) & df.duration.isin(VIDEOMME_DURATIONS)]
    rows = []
    for r in df.itertuples():
        # options arrive as "A. text" -> keep just the text, the letter is implied by position
        opts = [o.split(". ", 1)[1] if o[:2] in {f"{l}." for l in LETTERS} else o for o in r.options]
        rows.append(dict(
            qid=f"vmme_{r.question_id}", dataset="videomme", category=task_to_cat[r.task_type], task=r.task_type,
            video=f"{r.videoID}.mp4", question=r.question, options=opts, answer=r.answer.strip(),
            start=None, end=None, duration=r.duration,
        ))
    return pd.DataFrame(rows)


loaders = {"mvbench": load_mvbench, "videomme": load_videomme}
ann = pd.concat([loaders[d]() for d in DATASETS], ignore_index=True)
print(len(ann), "candidate questions")
ann.groupby(["dataset", "category", "task"]).size().rename("questions").to_frame()

## 4. Zip index + subset sampling

Each zip's central directory is read once (local file if mirrored, otherwise remote range reads) and cached in
`data/subset/zip_index_<dataset>.json`. Every question is then resolved to `(zip_file, zip_member, bytes)`.

In [ ]:
_fs = HfFileSystem()


def zip_source(dataset, zip_file):
    local = RAW_ROOT / dataset / zip_file
    return local if local.exists() else f"datasets/{REPOS[dataset]}/{zip_file}"


def open_zip(dataset, zip_file):
    src = zip_source(dataset, zip_file)
    fh = open(src, "rb") if isinstance(src, Path) else _fs.open(src, "rb", block_size=16 * 2**20)
    return zipfile.ZipFile(fh)


def zip_files_for(dataset):
    if dataset == "mvbench":
        tasks = [t for ts in MVBENCH_TASKS.values() for t in ts]
        names = {MVBENCH_PREFIX[t].split("/")[0] for t in tasks} | {"data0613"}
        return sorted(f"video/{n}.zip" for n in names)
    return [f"videos_chunked_{i:02d}.zip" for i in range(1, 21)]


def build_zip_index(dataset):
    cache = SUBSET_DIR / f"zip_index_{dataset}.json"
    index = json.loads(cache.read_text()) if cache.exists() else {}
    missing = [z for z in zip_files_for(dataset) if z not in index]
    for zf in tqdm(missing, desc=f"indexing {dataset} zips"):
        with open_zip(dataset, zf) as z:
            index[zf] = {i.filename: i.file_size for i in z.infolist() if not i.is_dir()}
        cache.write_text(json.dumps(index))
    return index


def resolve_members(df, dataset, index):
    member_to_zip = {m: (zf, size) for zf, members in index.items() for m, size in members.items()}
    out = []
    for r in df.itertuples():
        if dataset == "mvbench":
            prefix = MVBENCH_PREFIX[r.task]
            cands = [prefix + r.video, "data0613/" + prefix + r.video]
        else:
            cands = ["data/" + r.video]
        hit = next((c for c in cands if c in member_to_zip), None)
        out.append((*member_to_zip[hit], hit) if hit else (None, None, None))
    df = df.copy()
    df["zip_file"], df["video_bytes"], df["zip_member"] = zip(*out)
    return df


resolved = []
for d in DATASETS:
    part = resolve_members(ann[ann.dataset == d], d, build_zip_index(d))
    n_bad = part.zip_member.isna().sum()
    if n_bad:
        print(f"WARNING {d}: {n_bad} questions have no video in the zips and are dropped")
    resolved.append(part.dropna(subset=["zip_member"]).astype({"video_bytes": "int64"}))
ann = pd.concat(resolved, ignore_index=True)
print(len(ann), "questions with a resolvable video")

In [ ]:
def sample_category(df, n, seed):
    # Round-robin over the category's tasks; a video is used at most once per category.
    pools = {t: g.sample(frac=1, random_state=seed).to_dict("records") for t, g in df.groupby("task")}
    order, picked, seen = sorted(pools), [], set()
    while len(picked) < n and any(pools.values()):
        for t in order:
            while pools[t]:
                r = pools[t].pop()
                if r["zip_member"] not in seen:
                    seen.add(r["zip_member"])
                    picked.append(r)
                    break
            if len(picked) >= n:
                break
    return picked


def num_or_none(v):
    return None if v is None or pd.isna(v) else round(float(v), 3)


def frame_key(r):
    stem = Path(r["zip_member"]).stem
    if num_or_none(r["start"]) is not None:
        stem += f"__{r['start']:.2f}-{r['end']:.2f}"
    return stem


subset = []
for (d, c), g in ann.groupby(["dataset", "category"]):
    picked = sample_category(g, N_PER_CATEGORY, SEED)
    if len(picked) < N_PER_CATEGORY:
        print(f"WARNING {d}/{c}: only {len(picked)} distinct videos available")
    subset.extend(picked)

subset = pd.DataFrame(subset)
subset["video_path"] = [f"videos/{d}/{m}" for d, m in zip(subset.dataset, subset.zip_member)]
subset["frame_key"] = [frame_key(r) for r in subset.to_dict("records")]

# Frame keys must be unique per dataset (same key == same video segment == same frames, which is fine)
dup = subset.groupby(["dataset", "frame_key"]).video_path.nunique()
assert (dup == 1).all(), dup[dup > 1]


def json_clean(v):
    return None if isinstance(v, float) and np.isnan(v) else v


with open(SUBSET_DIR / "manifest.jsonl", "w", encoding="utf-8") as f:
    for r in subset.to_dict("records"):
        f.write(json.dumps({k: json_clean(v) for k, v in r.items()}, ensure_ascii=False) + "\n")
subset.assign(options=subset.options.map(json.dumps)).to_csv(SUBSET_DIR / "manifest.csv", index=False)

videos = subset.drop_duplicates(["dataset", "zip_member"])
print(f"{len(subset)} questions | {len(videos)} unique videos | "
      f"{videos.video_bytes.sum() / 1e9:.2f} GB to download")
pd.crosstab([subset.dataset, subset.task], subset.category, margins=True)

### Calibration split

`CALIB_PER_CATEGORY` extra questions per category and dataset, drawn with a different seed from videos that are
**not** used anywhere in the test subset. The proposed method tunes its confidence threshold on these only, so
the 350 test questions stay untouched. Saved to `data/subset/manifest_calib.jsonl`.

In [ ]:
calib = pd.DataFrame()
if CALIB_PER_CATEGORY:
    used = set(zip(subset.dataset, subset.zip_member))
    pool = ann[[(d, m) not in used for d, m in zip(ann.dataset, ann.zip_member)]]
    rows = []
    for (d, c), g in pool.groupby(["dataset", "category"]):
        picked = sample_category(g, CALIB_PER_CATEGORY, SEED + 1)
        if len(picked) < CALIB_PER_CATEGORY:
            print(f"WARNING calib {d}/{c}: only {len(picked)} unused videos available")
        rows.extend(picked)
    calib = pd.DataFrame(rows)
    calib["video_path"] = [f"videos/{d}/{m}" for d, m in zip(calib.dataset, calib.zip_member)]
    calib["frame_key"] = [frame_key(r) for r in calib.to_dict("records")]
    # different zips can hold files with the same name -> drop calib rows whose frame folder would collide
    taken = dict(zip(zip(subset.dataset, subset.frame_key), subset.video_path))
    clash = [taken.get((d, k), p) != p for d, k, p in zip(calib.dataset, calib.frame_key, calib.video_path)]
    if any(clash):
        print(f"dropping {sum(clash)} calib rows with a frame_key clash")
        calib = calib[[not x for x in clash]]
    with open(SUBSET_DIR / "manifest_calib.jsonl", "w", encoding="utf-8") as f:
        for r in calib.to_dict("records"):
            f.write(json.dumps({k: json_clean(v) for k, v in r.items()}, ensure_ascii=False) + "\n")
    calib.assign(options=calib.options.map(json.dumps)).to_csv(SUBSET_DIR / "manifest_calib.csv", index=False)

all_rows = pd.concat([subset.assign(split="test"), calib.assign(split="calib")], ignore_index=True)
videos = all_rows.drop_duplicates(["dataset", "zip_member"])
print(f"test {len(subset)} + calib {len(calib)} questions | {len(videos)} unique videos | "
      f"{videos.video_bytes.sum() / 1e9:.2f} GB total (already-downloaded files are skipped)")
pd.crosstab([all_rows.split, all_rows.dataset], all_rows.category, margins=True)

## 5. Download the subset videos

Only the sampled members are streamed out of each zip. Finished files are skipped on re-run (size check), and
partial files are written as `.part` first, so the cell can safely be interrupted and resumed.

In [ ]:
_tls = threading.local()


def _thread_zip(dataset, zip_file):
    cache = _tls.__dict__.setdefault("zips", {})
    key = (dataset, zip_file)
    if key not in cache:
        cache[key] = open_zip(dataset, zip_file)
    return cache[key]


def download_video(r, retries=3):
    dest = DATA_ROOT / r["video_path"]
    if dest.exists() and dest.stat().st_size == r["video_bytes"]:
        return "skipped"
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_name(dest.name + ".part")
    for attempt in range(1, retries + 1):
        try:
            z = _thread_zip(r["dataset"], r["zip_file"])
            with z.open(r["zip_member"]) as src, open(tmp, "wb") as out:
                shutil.copyfileobj(src, out, 8 * 2**20)
            tmp.replace(dest)
            return "downloaded"
        except Exception:
            _tls.__dict__.get("zips", {}).pop((r["dataset"], r["zip_file"]), None)
            if attempt == retries:
                raise


status = Counter()
failed = []
with ThreadPoolExecutor(DOWNLOAD_WORKERS) as pool:
    futs = {pool.submit(download_video, r): r for r in videos.to_dict("records")}
    for fut in tqdm(as_completed(futs), total=len(futs), desc="videos"):
        try:
            status[fut.result()] += 1
        except Exception as e:
            failed.append((futs[fut]["video_path"], repr(e)))
print(dict(status), "| failed:", len(failed))
for p, e in failed[:10]:
    print("  ", p, e)

## 6. Extract frames at the per-dataset `FPS`

Frames are sampled by timestamp: target times are `start, start + 1/FPS, start + 2/FPS, ...` and each target
takes the first decoded frame at or after it. So `FPS = 1` gives one frame per second of video, and `FPS` above
the native frame rate just keeps every frame.

For bounded MVBench questions only the `[start, end]` segment is kept. If the bounds fall outside the video,
the whole video is used and `bounds_ignored` is set.

Frames go to `data/frames/fps_<n>/<dataset>/`, so changing `FPS` for one dataset never overwrites the other.

`frames.json` is written last and doubles as a completion marker, so re-running skips finished videos.

In [ ]:
def fps_dirname(fps):
    return "fps_" + f"{fps:g}".replace(".", "p")


def _save_jpg(path, img, max_side, quality):
    if max_side and max(img.shape[:2]) > max_side:
        s = max_side / max(img.shape[:2])
        img = cv2.resize(img, (round(img.shape[1] * s), round(img.shape[0] * s)), interpolation=cv2.INTER_AREA)
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    if not ok:
        raise IOError(f"jpeg encode failed for {path}")
    buf.tofile(str(path))      # tofile handles non-ASCII Windows paths, cv2.imwrite does not
    return img.shape[1], img.shape[0]


def _sample(video_path, out_dir, fps, start, end, max_side, quality):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"cannot open {video_path}")
    native_fps = cap.get(cv2.CAP_PROP_FPS)
    if not (0 < native_fps < 1000):
        native_fps = 30.0
    t0, t1 = (start or 0.0), (end if end is not None else float("inf"))
    half = 0.5 / native_fps
    next_t, idx, frames, size = t0, -1, [], None
    while True:
        if not cap.grab():
            break
        idx += 1
        t = idx / native_fps
        if t > t1 + half:
            break
        if t + half < next_t:
            continue
        ok, img = cap.retrieve()
        if not ok:
            continue
        name = f"frame_{len(frames):05d}.jpg"
        size = _save_jpg(out_dir / name, img, max_side, quality)
        frames.append({"i": len(frames), "t": round(t, 3), "src_frame": idx, "file": name})
        while next_t <= t + half:
            next_t += 1.0 / fps
    cap.release()
    return frames, native_fps, idx + 1, size


def extract_frames(r, fps=None, max_side=FRAME_MAX_SIDE, quality=JPEG_QUALITY):
    fps = fps or fps_for(r["dataset"])
    out_dir = FRAMES_ROOT / fps_dirname(fps) / r["dataset"] / r["frame_key"]
    meta_path = out_dir / "frames.json"
    if meta_path.exists():
        return "skipped"
    if out_dir.exists():
        shutil.rmtree(out_dir)                 # leftovers of an interrupted run
    out_dir.mkdir(parents=True)
    video = DATA_ROOT / r["video_path"]
    use_bounds = USE_MVBENCH_BOUNDS and num_or_none(r["start"]) is not None
    start, end = (num_or_none(r["start"]), num_or_none(r["end"])) if use_bounds else (None, None)
    frames, native_fps, n_decoded, size = _sample(video, out_dir, fps, start, end, max_side, quality)
    bounds_ignored = False
    if not frames and use_bounds:
        frames, native_fps, n_decoded, size = _sample(video, out_dir, fps, None, None, max_side, quality)
        bounds_ignored = True
    if not frames:
        raise IOError(f"no frames decoded from {video}")
    meta = dict(dataset=r["dataset"], frame_key=r["frame_key"], video_path=r["video_path"], fps=fps,
                native_fps=native_fps, decoded_frames=n_decoded, video_seconds=round(n_decoded / native_fps, 3),
                start=start, end=end, bounds_ignored=bounds_ignored, width=size[0], height=size[1],
                n_frames=len(frames), frames=frames)
    meta_path.write_text(json.dumps(meta, indent=1))
    return "extracted"


jobs = all_rows.drop_duplicates(["dataset", "frame_key"])        # test + calibration
jobs = jobs[[(DATA_ROOT / p).exists() for p in jobs.video_path]]
status, failed = Counter(), []
with ThreadPoolExecutor(EXTRACT_WORKERS) as pool:
    futs = {pool.submit(extract_frames, r): r for r in jobs.to_dict("records")}
    for fut in tqdm(as_completed(futs), total=len(futs), desc=f"frames @ {FPS} fps"):
        try:
            status[fut.result()] += 1
        except Exception as e:
            failed.append((futs[fut]["frame_key"], repr(e)))
print(dict(status), "| failed:", len(failed))
for k, e in failed[:10]:
    print("  ", k, e)

## 7. Frame statistics + visual check

In [ ]:
def load_meta(r, fps=None):
    p = FRAMES_ROOT / fps_dirname(fps or fps_for(r["dataset"])) / r["dataset"] / r["frame_key"] / "frames.json"
    return json.loads(p.read_text()) if p.exists() else None


stats = []
for r in subset.to_dict("records"):
    m = load_meta(r)
    stats.append(dict(dataset=r["dataset"], category=r["category"], has_frames=m is not None,
                      n_frames=m["n_frames"] if m else np.nan, seconds=m["video_seconds"] if m else np.nan,
                      bounds_ignored=m["bounds_ignored"] if m else False))
stats = pd.DataFrame(stats)
print("questions without frames:", (~stats.has_frames).sum(), "| bounds ignored:", stats.bounds_ignored.sum())
stats.groupby(["dataset", "category"]).agg(
    questions=("has_frames", "size"), video_s_mean=("seconds", "mean"),
    frames_mean=("n_frames", "mean"), frames_min=("n_frames", "min"), frames_max=("n_frames", "max"),
).round(1)

In [ ]:
import matplotlib.pyplot as plt

r = subset.sample(1, random_state=None).iloc[0].to_dict()
m = load_meta(r)
if m is None:
    print("no frames for", r["qid"])
else:
    show = m["frames"][:: max(1, len(m["frames"]) // 12)][:12]
    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    for ax in axes.flat:
        ax.axis("off")
    for ax, f in zip(axes.flat, show):
        img = cv2.imread(str(FRAMES_ROOT / fps_dirname(fps_for(r["dataset"])) / r["dataset"] / r["frame_key"] / f["file"]))
        ax.imshow(img[:, :, ::-1])
        ax.set_title(f"t = {f['t']:.1f}s", fontsize=9)
    opts = "  ".join(f"({LETTERS[i]}) {o}" for i, o in enumerate(r["options"]))
    fig.suptitle(f"[{r['dataset']} / {r['category']} / {r['task']}]  {r['question']}\n{opts}   ->  answer {r['answer']}",
                 fontsize=10)
    plt.tight_layout()
    plt.show()